# Analysis on City Sprawl Through the Lens of Winnipeg Building Permits

## EDA Notebook

Aylie Lapierre Nagler <br>
COMP 2040 <br>
April 21, 2026

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import your helper module
import sys
sys.path.insert(0, '..')
from src.helpers import *

In [3]:
# Load data
df = pd.read_csv('../data/building-permits.csv', dtype={23: str, 24: str, 25: str, 26: str})

## Initial Exploration

### Mixed Data Types

Loading the dataset shows there are 4 columns with mixed datatypes. This will be explored first:

*Check the dtypes:*

In [23]:
dtypes = [('23: adding_secondary_suite', df['adding_secondary_suite'].dtypes), ('24: removing_secondary_suite', df['removing_secondary_suite'].dtypes),
           ('25: pool_type', df['pool_type'].dtypes), ('26: type_of_structure', df['type_of_structure'].dtypes)]
dtypes

[('23: adding_secondary_suite', dtype('O')),
 ('24: removing_secondary_suite', dtype('O')),
 ('25: pool_type', dtype('O')),
 ('26: type_of_structure', dtype('O'))]

Each of these columns store data as an object. The unique values will be explored next to determine whether this column should be stored differently:

In [22]:
unique_values = [('23: adding_secondary_suite', df['adding_secondary_suite'].unique()), ('24: removing_secondary_suite', df['removing_secondary_suite'].unique()),
           ('25: pool_type', df['pool_type'].unique()), ('26: type_of_structure', df['type_of_structure'].unique())]

for column_name, value in unique_values:
    print(column_name, value)

23: adding_secondary_suite [nan 'No' 'Yes']
24: removing_secondary_suite [nan 'No' 'Yes']
25: pool_type [nan 'In-ground' 'Above ground' 'Other']
26: type_of_structure [nan 'Shed' 'Other' 'Gazebo' 'Workshop' 'Farm Building' 'Greenhouse']


These columns are likely reporting mixed datatypes due to the 'NaN' values. These will all be converted to objects in the next notebook, though columns 23 and 24 may be converted to binary at a later step.

### Data Shape

In [3]:
df.shape

(157928, 35)

*35 Columns*<br>
*157928 Rows*

### Data Information

This dataset has 35 columns. As such, the outputs of functions that return information for each column are truncated due to length. Instead of running them directly in the notebook they are nested in helper functions which return a cleaner output containing the same information:

**Data Types**

In [ ]:
# Get columns and data type in table format (2 cols)
split_columns(df)


,Column Names,Data Types,Column Names (cont.),Data Types (cont.)
0,issue_date,object,dwelling_units_lost,int64
1,permit_number,object,location,object
2,parent_permit_number,object,x_coordinate_nad83,float64
3,permit_group,object,y_coordinate_nad83,float64
4,permit_type,object,includes_secondary_suite,object
5,sub_type,object,adding_secondary_suite,object
6,work_type,object,removing_secondary_suite,object
7,street_number,float64,pool_type,object
8,street_name,object,type_of_structure,object
9,street_type,object,ward,object


In [5]:
# Use .dtypes and .value_counts() to get count of data types
df.dtypes.value_counts()

object     29
float64     4
int64       2
Name: count, dtype: int64

*object:* 29 <br>
*float:* 4 <br>
*integer:* 2

**NaN Counts**

In [ ]:
# Call nan_count() function to get count of NaN values in each column
nan_count(df)

,Column Names,NaN Count,Column Names (cont.),NaN Count (cont.)
0,issue_date,0,dwelling_units_lost,0.0
1,permit_number,0,location,355.0
2,parent_permit_number,145795,x_coordinate_nad83,355.0
3,permit_group,0,y_coordinate_nad83,355.0
4,permit_type,0,includes_secondary_suite,111342.0
5,sub_type,0,adding_secondary_suite,144047.0
6,work_type,2,removing_secondary_suite,144047.0
7,street_number,290,pool_type,156825.0
8,street_name,272,type_of_structure,156967.0
9,street_type,1126,ward,88.0


*Observations about NaN counts:*
1. counts for 'location', 'x_coordinate_nad83', 'y_coordinate_nad83', and 'point' are the same.
2. counts for 'adding_secondary_suite' and 'removing_secondary_suite' are the same.
3. counts for 'unit_type' and 'unit_number' are the same.
4. counts for 'neighbourhood_number' and 'neighbourhood_name' are the same.
5. counts  for 'work_type' and 'address' are the same.

There are three categories of NaN values present in this data: 
1. Expected (where the NaN can be interpreted as 'N/A')
2. Innocuous (NaN value is not a cause for concern)
3. Potentially harmful (NaNs that need to be handled and may signal data quality issues - ex: NaN in what would have been a primary key column)

The table below shows where the columns in this dataset fall within thesed three categories:

| Expected (N/A) | Innocuous | Potentially Harmful |
|---|---|---|
| parent_permit_number | ward | work_type |
| street_direction | | street_number |
| unit_type | | street_name |
| unit_number | | street_type |
| applicant_business_name | | neighbourhood_name |
| pool_type | | neighbourhood_number |
| | | location |
| | | x_coordinate_nad83 |
| | | y_coordinate_nad83 |
| | | type_of_structure |
| | | final_date |
| | | point |
| | | address |

Many of these can be filled in at a later state. For example, 'street_number', 'street_name', 'street_type', and 'street_direction' are all atomic versions of the 'address' column. While the atomic versions have many missing values, the address column only has 2:

In [7]:
# Display NaN count for columns relating to address
nan_count(df[['street_number', 'street_name', 'street_type', 'street_direction', 'address']])

,Column Names,NaN Count,Column Names (cont.),NaN Count (cont.)
0,street_number,290,street_direction,149811.0
1,street_name,272,address,2.0
2,street_type,1126,,


In [8]:
# Display a row of columns pertaining to address to show relation
df[['street_number', 'street_name', 'street_type', 'street_direction', 'address']].iloc[7]

street_number             94.0
street_name               Hill
street_type                 ST
street_direction           NaN
address             94 Hill ST
Name: 7, dtype: object

### Describe Data

In [9]:
df.describe()

,street_number,neighbourhood_number,dwelling_units_created,dwelling_units_lost,x_coordinate_nad83,y_coordinate_nad83
count,157638.000000,157924.000000,157928.000000,157928.000000,157573.000000,1.575730e+05
mean,537.101771,3.369311,0.462686,0.029267,633226.429631,5.526123e+06
std,720.802153,1.434020,5.302709,0.432818,5186.595367,5.706253e+03
min,1.000000,1.103000,0.000000,0.000000,619062.426042,5.510167e+06
25%,79.000000,2.221000,0.000000,0.000000,629852.180850,5.521825e+06
50%,252.000000,3.302000,0.000000,0.000000,632847.066700,5.526734e+06
75%,720.000000,4.422000,0.000000,0.000000,636564.714700,5.530064e+06
max,7740.000000,5.674000,395.000000,91.000000,646508.814691,5.538939e+06


Columns like 'street_number' and 'neighbourhood_number' don't have statistical relevance unless the geography is organized numerically. At this stage the coordinates also lack meaningdul statistical measures since there are so many missing values. These missing values may be able to be filled in later at which point these stats will be more complete. At this stage, 'dwelling_units_created' and 'dwelling_units_lost' are the most meaningful stats.

Observations from meaningful columns:
1. dwelling_units_created
    - mean of 0.46: on avg, less than half a dwelling unit was created per permit
    - median of 0: most permits had 0 units created
    - std of 5.30: this column sees a lot of variability (compared to mean of 0.46) 
    - 75 percentile at 0: less than 25% of permits created a dwelling
    - max of 395: compared to the max of 'dwelling_units_lost' (92), it is more common for large scale building of dwellings than large scale demolitions.
2. dwelling_units_lost
    - mean of 0.03: on avg, about 3% of permits removed dwellings
    - median of 0: most permits had 0 units removed
    - std of 0.43: this column sees some variability (0.43 is still large compared to mean of 0.03)
    - 75 percentile at 0: less than 25% of permits removed a dwelling

## Analytical Questions

1. Are there hotspots for certain types of construction? If so, why?
2. Are there any temporal patterns indicating the geographical direction of growth in the city?
3. There are a lot of null values for end dates, are there a lot of projects that don't get completed or does this reflect a lack of followup on permits? If it is the former, are any companies repeatedly not completing projects?
4. Can this dataset highlight residential neighbourhoods whose houses tend to require more work on average?
5. What is the distribution of types of permits issued and how has it changed over time?
6. Are there economic indicators in this data? For example, if all other data disappeared and this was the only data we had to work from, could we detect upticks and downturns in the economy based on the amount and frequency of permits being requested?